Template: bootstrap → config → `run_cli_streaming`. Set `CONTEST` to your key.

Some contests expose **`train_and_submit`** / **`tune_and_submit`** on `notebook_commands`; the runner uses them when `PIPELINE` matches and `getattr(cmds, ...)` is present.


In [ ]:
# Bootstrap Cell: paths + framework + infra notebook API
#
# Must insert scripts/ before importing path_bootstrap (same resolution as
# path_bootstrap.resolve_notebook_scripts_path in the package).

import os
import sys

CONTEST = "csiro"  # Change for your competition key.
SCRIPTS_PACKAGE = "kaggle-ml-comp-scripts"

is_kaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
if is_kaggle:
    scripts_candidates = [
        f"/kaggle/input/datasets/mcusac/{SCRIPTS_PACKAGE}/scripts",
        f"/kaggle/input/{SCRIPTS_PACKAGE}/scripts",
    ]
else:
    scripts_candidates = [
        f"../input/{SCRIPTS_PACKAGE}/scripts",
        f"../{SCRIPTS_PACKAGE}/scripts",
    ]

scripts_path = next((p for p in scripts_candidates if os.path.isdir(p)), None)
if scripts_path is None:
    raise FileNotFoundError(
        "Could not locate kaggle-ml-comp-scripts/scripts. Checked:\n"
        + "\n".join(f" - {p}" for p in scripts_candidates)
    )

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from path_bootstrap import bootstrap_notebook_environment

scripts_path = bootstrap_notebook_environment(contest=CONTEST, scripts_package=SCRIPTS_PACKAGE)

from layers.layer_1_competition.level_0_infra.level_1 import (
    get_notebook_commands_module,
    list_contests_with_notebook_commands,
    run_cli_streaming,
)

available_contests = list_contests_with_notebook_commands()

try:
    cmds = get_notebook_commands_module(CONTEST)
except ValueError as exc:
    raise ValueError(
        f"Contest '{CONTEST}' is not registered for notebook commands. "
        f"Available contests: {available_contests}. "
        "Add a contest registration with register_notebook_commands_module(...), "
        "or change CONTEST to an available key."
    ) from exc

print("✅ Setup complete. Run Config Cell next.")
print(f"   Contest: {CONTEST}")
print(f"   Scripts path: {scripts_path}")
print(f"   Contests with notebook_commands registered: {available_contests}")


In [ ]:
# Config Cell
PIPELINE = "submission"
SMOKE_MAX_TARGETS = 3

if PIPELINE == "validate_data":
    VALIDATE_LOG_FILE = None

if PIPELINE in ("train", "train_and_submit"):
    TRAIN_PIPELINE = "end_to_end"
    TRAIN_MODELS = ["baseline_approx"]

if PIPELINE in ("tune", "tune_and_submit"):
    TUNE_MODEL = "baseline_approx"
    TUNE_SEARCH_TYPE = "quick"

if PIPELINE in ("submission", "train_and_submit", "tune_and_submit"):
    SUBMISSION_PIPELINE = "single"
    SUBMISSION_MODELS = ["baseline_approx"]
    SUBMISSION_ENSEMBLE_WEIGHTS = None
    USE_VALIDATION_FOR_STACKING = True

print("Config ready")
print(f"   PIPELINE: {PIPELINE}")
if PIPELINE == "validate_data":
    print(f"   VALIDATE_LOG_FILE: {VALIDATE_LOG_FILE}")
if PIPELINE in ("train", "train_and_submit"):
    print(f"   TRAIN_PIPELINE: {TRAIN_PIPELINE}")
    print(f"   TRAIN_MODELS: {TRAIN_MODELS}")
if PIPELINE in ("tune", "tune_and_submit"):
    print(f"   TUNE_MODEL: {TUNE_MODEL}")
    print(f"   TUNE_SEARCH_TYPE: {TUNE_SEARCH_TYPE}")
if PIPELINE in ("submission", "train_and_submit", "tune_and_submit"):
    print(f"   SUBMISSION_PIPELINE: {SUBMISSION_PIPELINE}")
    print(f"   SUBMISSION_MODELS: {SUBMISSION_MODELS}")
print(f"   SMOKE_MAX_TARGETS: {SMOKE_MAX_TARGETS}")

In [ ]:
if PIPELINE == "validate_data":
    cmd = cmds.build_validate_data_command(
        max_targets=SMOKE_MAX_TARGETS,
        log_file=VALIDATE_LOG_FILE,
    )
    description = f"{CONTEST} validate_data (max_targets={SMOKE_MAX_TARGETS})"

elif PIPELINE == "train":
    cmd = cmds.build_train_command(train_mode=TRAIN_PIPELINE, models=TRAIN_MODELS)
    description = f"Training {TRAIN_PIPELINE} models: {TRAIN_MODELS}"

elif PIPELINE == "tune":
    cmd = cmds.build_tune_command(
        model=TUNE_MODEL,
        search_type=TUNE_SEARCH_TYPE,
        max_targets=SMOKE_MAX_TARGETS,
    )
    description = f"Tuning {TUNE_MODEL} with {TUNE_SEARCH_TYPE} search"

elif PIPELINE == "submission":
    cmd = cmds.build_submit_command(
        strategy=SUBMISSION_PIPELINE,
        models=SUBMISSION_MODELS,
        max_targets=SMOKE_MAX_TARGETS,
        ensemble_weights=SUBMISSION_ENSEMBLE_WEIGHTS,
        use_validation_for_stacking=USE_VALIDATION_FOR_STACKING,
    )
    description = f"Submission {SUBMISSION_PIPELINE} models={SUBMISSION_MODELS}"

elif PIPELINE == "train_and_submit":
    fn = getattr(cmds, "build_train_and_submit_command", None)
    if fn is None:
        raise ValueError(f"{CONTEST}: notebook_commands has no build_train_and_submit_command")
    cmd = fn(
        train_mode=TRAIN_PIPELINE,
        models=TRAIN_MODELS,
        strategy=SUBMISSION_PIPELINE,
        max_targets=SMOKE_MAX_TARGETS,
        ensemble_weights=SUBMISSION_ENSEMBLE_WEIGHTS,
        use_validation_for_stacking=USE_VALIDATION_FOR_STACKING,
    )
    description = f"train_and_submit train={TRAIN_PIPELINE} submit={SUBMISSION_PIPELINE}"

elif PIPELINE == "tune_and_submit":
    fn = getattr(cmds, "build_tune_and_submit_command", None)
    if fn is None:
        raise ValueError(f"{CONTEST}: notebook_commands has no build_tune_and_submit_command")
    cmd = fn(
        model=TUNE_MODEL,
        search_type=TUNE_SEARCH_TYPE,
        strategy=SUBMISSION_PIPELINE,
        models=SUBMISSION_MODELS,
        max_targets=SMOKE_MAX_TARGETS,
        ensemble_weights=SUBMISSION_ENSEMBLE_WEIGHTS,
        use_validation_for_stacking=USE_VALIDATION_FOR_STACKING,
    )
    description = f"tune_and_submit tune={TUNE_MODEL} submit={SUBMISSION_PIPELINE}"

else:
    raise ValueError(f"Unknown PIPELINE: {PIPELINE}")

returncode, stdout_lines = run_cli_streaming(cmd, description=description, keep_last_n=200)

if returncode != 0:
    raise RuntimeError(f"{PIPELINE} pipeline failed. See output above.\n" + "\n".join(stdout_lines[-50:]))
